# MLP from scratch
Yahia Chemlali

Dataset : [Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist)

---

Le MLP a été abordé en ouverture dans le cours d'IAS (Introduction Apprentissage Statistique).

Les ressources utile qui m'ont aidé à comprendre : https://web.stanford.edu/~jurafsky/slp3/ et https://rtavenar.github.io/deep_book/book_fr.pdf

Le but c'est de coder un perceptron multi couches sans pytorch/tensorflow, juste numpy. On va le tester sur Fashion-MNIST : 70 000 images 28x28 de vetements (t-shirts, chaussures, sacs...), 10 classes. L'important pour moi c'est de coder la classe MLP et de comprendre comment marchent les fonctions d'activation et la backpropagation, bref les mathematiques derriere.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

CLASS_NAMES = [
    'T-shirt', 'Pantalon', 'Pull', 'Robe', 'Manteau',
    'Sandale', 'Chemise', 'Basket', 'Sac', 'Bottine'
]

## Fonctions d'activation

On utilise ReLU pour les couches cachées et Softmax pour la sortie.

ReLU c'est simple : f(x) = max(0, x). Ca met à zero tout ce qui est negatif. Ca marche mieux que sigmoid sur des reseaux profonds parce que le gradient ne disparait pas.

Softmax transforme un vecteur de scores en distribution de probabilites. Pour eviter les overflow avec exp(), on soustrait le max avant — ca change pas le resultat mais ca stabilise les calculs.

In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    # return z > 0
    return (z > 0).astype(float)

def softmax(z):
    # e = np.exp(z)
    e = np.exp(z - np.max(z, axis=0, keepdims=True))
    return e / np.sum(e, axis=0, keepdims=True)

## La loss

On utilise la cross-entropy. Pour une classification multi-classe c'est la loss qui va avec softmax. Elle mesure à quel point la distribution predite est proche de la vraie classe.

Formule : L = -sum(y * log(y_pred)) / m

Le `y` c'est le one-hot (0 partout sauf 1 sur la bonne classe), donc le sum elimine toutes les autres classes — seule la probabilite de la bonne classe compte. Le epsilon (1e-12) c'est pour eviter log(0) si le modele met 0% sur une classe.

In [ ]:
def cross_entropy(pred, vrai):
    m = vrai.shape[1]
    return -np.sum(vrai * np.log(pred + 1e-12)) / m

## La classe MLP

C'est là où tout se passe. À la construction on initialise les poids avec He initialization — j'ai essayé sans et les activations partent dans tous les sens dès les premières itérations, donc c'est important surtout avec ReLU.

Ensuite il y a trois méthodes : `forward` qui calcule les activations couche par couche et stocke tout dans `mem` (on en a besoin pour la backprop après), `backward` qui calcule les gradients avec la chain rule, et `update` qui fait la descente de gradient.

Pour l'entrainement j'utilise des mini-batchs de 64 exemples. J'ai essayé en full-batch sur les 56k samples, c'est interminable.

In [ ]:
class MLP:
    def __init__(self, couches, acts, lr=0.01):
        self.acts = acts
        self.lr = lr
        self.L = len(couches) - 1
        rng = np.random.default_rng(42)
        self.W, self.b = {}, {}
        for i in range(1, self.L + 1):
            n_in, n_out = couches[i-1], couches[i]
            scale = np.sqrt(2.0/n_in) if acts[i-1] == 'relu' else np.sqrt(1.0/n_in)
            self.W[i] = rng.normal(0, scale, (n_out, n_in))
            self.b[i] = np.zeros((n_out, 1))

    def forward(self, X):
        mem = {'A0': X}
        A = X
        for i in range(1, self.L + 1):
            Z = self.W[i] @ A + self.b[i]
            mem['Z' + str(i)] = Z
            A = softmax(Z) if self.acts[i-1] == 'softmax' else relu(Z)
            mem['A' + str(i)] = A
        return A, mem

    def backward(self, y, mem):
        m = y.shape[1]
        grads = {}

        ## dZ = mem['A' + str(self.L)] - y
        ## grads['dW' + str(self.L)] = dZ @ mem['A' + str(self.L-1)].T
        ## grads['db' + str(self.L)] = np.sum(dZ, axis=1, keepdims=True)

        dZ = mem['A' + str(self.L)] - y
        grads['dW' + str(self.L)] = (1/m) * dZ @ mem['A' + str(self.L-1)].T
        grads['db' + str(self.L)] = (1/m) * np.sum(dZ, axis=1, keepdims=True)
        for i in range(self.L-1, 0, -1):
            dA = self.W[i+1].T @ dZ
            dZ = dA * relu_deriv(mem['Z' + str(i)])
            grads['dW' + str(i)] = (1/m) * dZ @ mem['A' + str(i-1)].T
            grads['db' + str(i)] = (1/m) * np.sum(dZ, axis=1, keepdims=True)
        return grads

    def update(self, grads):
        for i in range(1, self.L + 1):
            self.W[i] -= self.lr * grads['dW' + str(i)]
            self.b[i] -= self.lr * grads['db' + str(i)]

    def train(self, X, y, iterations=30, batch_size=64, log=5):
        m = X.shape[1]
        losses = []
        accs = []
        for it in range(1, iterations+1):
            idx = np.random.permutation(m)
            batch_loss = 0
            nb = 0
            for start in range(0, m, batch_size):
                batch = idx[start:start+batch_size]
                Xb, yb = X[:, batch], y[:, batch]
                out, mem = self.forward(Xb)
                batch_loss += cross_entropy(out, yb)
                nb += 1
                grads = self.backward(yb, mem)
                self.update(grads)
            avg_loss = batch_loss / nb
            losses.append(avg_loss)
            out_full, _ = self.forward(X)
            acc = np.mean(np.argmax(out_full, 0) == np.argmax(y, 0))
            accs.append(acc)
            if it % log == 0 or it == 1:
                print("iteration " + str(it) + "  loss=" + str(round(avg_loss, 4)) + "  acc=" + str(round(acc * 100, 2)) + "%")
        return losses, accs

    def predict(self, X):
        out, _ = self.forward(X)
        return np.argmax(out, axis=0)


def one_hot(y, k):
    # oh = np.eye(k)[y]
    oh = np.zeros((k, len(y)))
    oh[y, np.arange(len(y))] = 1
    return oh

## Chargement du dataset

Fashion-MNIST c'est 70 000 images 28x28 en niveaux de gris, 10 classes de vetements. C'est plus interessant que MNIST chiffres parce que c'est plus dur et moins vu partout.

Le telechargement prend un peu de temps la premiere fois (~55 MB), ensuite c'est mis en cache.

On normalise entre 0 et 1 en divisant par 255 (valeur max des pixels). Sans ca le reseau converge beaucoup plus lentement. Sklearn donne les donnees en format (samples, features) mais le MLP attend (features, samples) pour les multiplications matricielles, donc on transpose.

In [ ]:
print("chargement de Fashion-MNIST...")
mnist = fetch_openml('Fashion-MNIST', version=1, as_frame=False)
X = mnist.data / 255.0
y = mnist.target.astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train, X_test = X_train.T, X_test.T

y_train_oh = one_hot(y_train, 10)
y_test_oh  = one_hot(y_test, 10)

print("train: " + str(X_train.shape[1]) + " samples")
print("test:  " + str(X_test.shape[1]) + " samples")
print("features: " + str(X_train.shape[0]))

## Un apercu des donnees

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[:, i].reshape(28, 28), cmap='gray')
    ax.set_title(CLASS_NAMES[y_train[i]], fontsize=7)
    ax.axis('off')
plt.suptitle('exemples du dataset')
plt.tight_layout()
plt.show()

## Entrainement

L'architecture c'est 784 entrées (28x28 pixels) -> 256 neurones -> 128 -> 10 sorties.

30 itérations donnent ~87% d'accuracy sur le test set. J'ai essayé d'aller plus loin mais passé 30 le gain est vraiment faible par rapport au temps que ça prend.

In [ ]:
net = MLP(
    couches=[784, 256, 128, 10],
    acts=['relu', 'relu', 'softmax'],
    lr=0.1
)

hist, accs = net.train(X_train, y_train_oh, iterations=30, batch_size=64, log=5)

## Evaluation sur le test set

In [ ]:
pred_test = net.predict(X_test)
acc_test = np.mean(pred_test == y_test)
print("accuracy sur le test set: " + str(round(acc_test * 100, 2)) + "%")

## Loss et accuracy

On trace les deux sur le meme plot pour voir comment elles evoluent ensemble.

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 4))

ax1.plot(hist, color='steelblue', label='loss')
ax1.set_xlabel('iteration')
ax1.set_ylabel('loss', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

ax2 = ax1.twinx()
ax2.plot(accs, color='coral', label='accuracy')
ax2.set_ylabel('accuracy', color='coral')
ax2.tick_params(axis='y', labelcolor='coral')

fig.legend(loc='center right', bbox_to_anchor=(0.88, 0.5))
plt.title('loss et accuracy par iteration')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Matrice de confusion

Pour voir ou le modele se trompe le plus. En general les erreurs se font entre classes visuellement proches, par exemple T-shirt / Chemise ou Basket / Sandale.

In [ ]:
cm = confusion_matrix(y_test, pred_test)
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap='Blues')
plt.title('matrice de confusion')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Quelques predictions

On affiche des images du test set avec la prediction du reseau. En vert si c'est bon, en rouge si c'est faux.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    img = X_test[:, i].reshape(28, 28)
    ax.imshow(img, cmap='gray')
    p = pred_test[i]
    v = y_test[i]
    color = 'green' if p == v else 'red'
    ax.set_title(CLASS_NAMES[p], color=color, fontsize=7)
    ax.axis('off')
plt.suptitle('vert=bon, rouge=erreur')
plt.tight_layout()
plt.show()